# Complete NLP Sentiment Analysis - Uganda TikTok Comments

This notebook performs:
- Text cleaning (emoji removal for frequency, emoji preservation for sentiment)
- Word frequency (top 15)
- N-gram analysis (bigrams/trigrams on selected topics)
- VADER sentiment analysis (compound score → positive/neutral/negative)
- Subjectivity (Fact/Opinion) with TextBlob
- Sentiment by keyword
- Temporal sentiment trend (if date column exists)
- Correlation between sentiment and likes (diggCount)
- Visualisations (pie and bar charts)
- Summary statistics table
- Export results to CSV
- Optional topic modeling (LDA) if scikit-learn is installed

In [ ]:
import json
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from collections import Counter
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.util import ngrams
from textblob import TextBlob
import nltk

# Download required NLTK data (run once)
nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)

In [ ]:
# Load data
file_path = "C:/Users/User/Documents/Desktop/Ugandan_comments.json"
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

if isinstance(data, list):
    comments_list = data
elif isinstance(data, dict) and 'comments' in data:
    comments_list = data['comments']
else:
    comments_list = data

df = pd.DataFrame(comments_list)
print(f"Loaded {df.shape[0]} comments, {df.shape[1]} columns")
df.head()

In [ ]:
# Slang mapping dictionary
slang_map = {
    'u': 'you', 'ur': 'your', 'r': 'are', 'yall': 'you all',
    'ya': 'you', 'yh': 'yeah', 'yep': 'yes', 'nope': 'no',
    'coz': 'because', 'wanna': 'want to', 'gonna': 'going to',
    'hv': 'have', 'fam': 'friend',
}

def clean_text_freq(text):
    """Cleaning for word frequency – removes emojis, URLs, mentions, hashtags, special spaces."""
    if pd.isna(text):
        return ''
    text = str(text).lower()
    # Remove bracketed content (sticker markers)
    text = re.sub(r'\[.*?\]', '', text)
    # Remove emojis
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text)
    # Remove special Unicode spaces
    special_spaces = re.compile(r'[\u115F\u1160\u3164\uFFA0\u200B-\u200F\u2028-\u202F\u205F\u3000]')
    text = special_spaces.sub(r' ', text)
    # Remove URLs, mentions, hashtags
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Slang mapping
    words = text.split()
    mapped_words = [slang_map.get(word, word) for word in words]
    return ' '.join(mapped_words)

def clean_text_sentiment(text):
    """Cleaning for sentiment – preserves emojis, removes only unnecessary characters."""
    if pd.isna(text):
        return ''
    text = str(text)
    # Remove bracketed content
    text = re.sub(r'\[.*?\]', '', text)
    # Remove special Unicode spaces
    special_spaces = re.compile(r'[\u115F\u1160\u3164\uFFA0\u200B-\u200F\u2028-\u202F\u205F\u3000]')
    text = special_spaces.sub(r' ', text)
    # Remove URLs and mentions (but keep emojis)
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r'@\w+', '', text)
    # Remove # symbol but keep the word (VADER understands words)
    text = re.sub(r'#', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Slang mapping (lowercase only for mapping)
    words = text.split()
    mapped_words = [slang_map.get(word.lower(), word) for word in words]
    return ' '.join(mapped_words)

df['clean_freq'] = df['text'].apply(clean_text_freq)
df['clean_sent'] = df['text'].apply(clean_text_sentiment)
print("Text cleaning completed.")

In [ ]:
# Word frequency (top 15)
stops = set(stopwords.words('english'))
all_words = ' '.join(df['clean_freq'].astype(str))
words = all_words.split()
filtered_words = [w.lower() for w in words if w.lower() not in stops]
freq_counter = Counter(filtered_words)

print("Top 15 most common words (stopwords removed):")
for word, count in freq_counter.most_common(15):
    print(f"  {word}: {count}")

In [ ]:
# N-gram analysis (bigrams and trigrams) on relevant topics
keywords = ['uganda', 'football', 'president', 'love', 'food', 'women', 'visit', 'language']
bigram_counts = Counter()
trigram_counts = Counter()

for text in df['clean_freq']:
    words = text.lower().split()
    if len(words) >= 2:
        for bg in ngrams(words, 2):
            bg_str = ' '.join(bg)
            if any(kw in bg_str for kw in keywords):
                bigram_counts[bg_str] += 1
    if len(words) >= 3:
        for tg in ngrams(words, 3):
            tg_str = ' '.join(tg)
            if any(kw in tg_str for kw in keywords):
                trigram_counts[tg_str] += 1

print("Top 10 bigrams:")
for bg, cnt in bigram_counts.most_common(10):
    print(f"  {bg}: {cnt}")

print("\nTop 10 trigrams:")
for tg, cnt in trigram_counts.most_common(10):
    print(f"  {tg}: {cnt}")

In [ ]:
# VADER sentiment analysis
sia = SentimentIntensityAnalyzer()
df['compound'] = df['clean_sent'].apply(lambda x: sia.polarity_scores(str(x))['compound'])
df['sentiment'] = df['compound'].apply(
    lambda x: 'positive' if x > 0.05 else ('negative' if x < -0.05 else 'neutral')
)

print("Sentiment distribution:")
print(df['sentiment'].value_counts())
print("\nPercentages:")
print(df['sentiment'].value_counts(normalize=True) * 100)

In [ ]:
# Subjectivity (Fact vs Opinion) with TextBlob
def get_subjectivity(text):
    try:
        return TextBlob(str(text)).sentiment.subjectivity
    except:
        return 0.0

df['subjectivity'] = df['clean_sent'].apply(get_subjectivity)
df['fact_opinion'] = df['subjectivity'].apply(
    lambda x: 'Fact' if x < 0.3 else ('Opinion' if x > 0.6 else 'Mixed')
)

print("Fact vs Opinion distribution:")
print(df['fact_opinion'].value_counts())
print("\nPercentages:")
print(df['fact_opinion'].value_counts(normalize=True) * 100)

In [ ]:
# Sentiment by keyword
kw_list = ['uganda', 'football', 'president', 'love', 'food', 'women', 'visit', 'language']
print("Average compound sentiment for each keyword:")
for kw in kw_list:
    mask = df['clean_freq'].str.contains(kw, case=False, na=False)
    if mask.sum() > 0:
        avg = df.loc[mask, 'compound'].mean()
        print(f"  '{kw}': {mask.sum()} comments, avg sentiment = {avg:.2f}")

In [ ]:
# Temporal analysis (if createTimeISO exists)
if 'createTimeISO' in df.columns:
    df['date'] = pd.to_datetime(df['createTimeISO']).dt.date
    daily_sent = df.groupby('date')['compound'].mean().reset_index()
    plt.figure(figsize=(10,4))
    plt.plot(daily_sent['date'], daily_sent['compound'], marker='o', linestyle='-')
    plt.title('Average Sentiment Over Time')
    plt.xlabel('Date')
    plt.ylabel('Compound Sentiment')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("'createTimeISO' column not found – skipping temporal analysis.")

In [ ]:
# Engagement correlation (diggCount vs sentiment)
if 'diggCount' in df.columns:
    print("Average likes per sentiment:")
    print(df.groupby('sentiment')['diggCount'].mean())
    print("\nCorrelation between sentiment (compound) and likes:")
    print(df[['compound', 'diggCount']].corr())
else:
    print("'diggCount' column not found – skipping engagement analysis.")

In [ ]:
# Visualisations (pie and bar charts)
counts = df['sentiment'].value_counts()
pos = counts.get('positive', 0)
neu = counts.get('neutral', 0)
neg = counts.get('negative', 0)
total = pos + neu + neg

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
axes[0].pie([pos, neu, neg], labels=['Positive', 'Neutral', 'Negative'],
            colors=['green', 'gold', 'red'], autopct='%1.1f%%')
axes[0].set_title('Overall Sentiment')

# Bar chart
axes[1].bar(['Positive', 'Neutral', 'Negative'], [pos, neu, neg],
            color=['green', 'gray', 'red'])
axes[1].set_ylabel('Number of Comments')
axes[1].set_title('Sentiment Counts')
for i, v in enumerate([pos, neu, neg]):
    axes[1].text(i, v + 20, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics table
strong_pos = (df['compound'] >= 0.5).sum()
strong_neg = (df['compound'] <= -0.5).sum()
avg_word_count = df['clean_freq'].str.split().str.len().mean()

summary = pd.DataFrame({
    'Metric': ['Total comments', 'Positive (%)', 'Neutral (%)', 'Negative (%)',
               'Strongly positive (>0.5)', 'Strongly negative (<-0.5)', 'Avg word count'],
    'Value': [
        len(df),
        f"{(pos/total)*100:.1f}%",
        f"{(neu/total)*100:.1f}%",
        f"{(neg/total)*100:.1f}%",
        strong_pos,
        strong_neg,
        f"{avg_word_count:.1f}"
    ]
})
print("Summary statistics:")
print(summary)

In [ ]:
# Export results to CSV
output_file = 'uganda_tiktok_sentiment_full.csv'
df.to_csv(output_file, index=False)
print(f"DataFrame saved to {output_file}")
print("Columns included: original text, cleaned texts, compound score, sentiment label, subjectivity, fact/opinion, etc.")

In [ ]:
# Optional Topic Modeling (LDA)
try:
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.decomposition import LatentDirichletAllocation

    docs = df['clean_freq'].fillna('').tolist()
    vectorizer = CountVectorizer(max_features=500, stop_words='english', min_df=5, max_df=0.8)
    dtm = vectorizer.fit_transform(docs)
    lda = LatentDirichletAllocation(n_components=5, random_state=42)
    lda.fit(dtm)
    feature_names = vectorizer.get_feature_names_out()
    print("\nTop 10 words per topic (LDA):")
    for i, topic in enumerate(lda.components_):
        top_words = [feature_names[j] for j in topic.argsort()[-10:][::-1]]
        print(f"Topic {i+1}: {', '.join(top_words)}")
except ImportError:
    print("scikit-learn not installed – LDA topic modeling skipped.")